# Projeto — Análise de Preços de Imóveis

**Dataset:** Housing Prices Dataset (King County)

Notebook organizado segundo as fases e questões do desafio.

## 🔹 FASE 1 — Limpeza e Padronização

### 1. Consolidação e Tipagem
Conversão das colunas para os tipos adequados: data para datetime, áreas de pés quadrados para metros quadrados e código postal para texto.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

#import plotly.io as pio
#pio.renderers.default = "vscode"

In [2]:
df = pd.read_csv('Housing.csv')

In [3]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
#Convertendo a Para Metros Quadrados

converter = ['sqft_living' , 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']
for i in converter:
    df[i]= df[i] * 0.092903

In [5]:
df['zipcode'] = df['zipcode'].astype(str)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21613 non-null  int64         
 1   date           21613 non-null  datetime64[us]
 2   price          21613 non-null  float64       
 3   bedrooms       21613 non-null  int64         
 4   bathrooms      21613 non-null  float64       
 5   sqft_living    21613 non-null  float64       
 6   sqft_lot       21613 non-null  float64       
 7   floors         21613 non-null  float64       
 8   waterfront     21613 non-null  int64         
 9   view           21613 non-null  int64         
 10  condition      21613 non-null  int64         
 11  grade          21613 non-null  int64         
 12  sqft_above     21613 non-null  float64       
 13  sqft_basement  21613 non-null  float64       
 14  yr_built       21613 non-null  int64         
 15  yr_renovated   21613 non-null 

### 2. Tratamento de Valores Ausentes
Verificação de valores nulos por coluna. O dataset não apresenta ausências, portanto nenhum registro precisou ser removido.

In [7]:
df.isna().sum()

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

### 3. Padronização de Variáveis Categóricas
Renomeação das colunas para português, padronizando a nomenclatura das variáveis de localização e tipo do imóvel.

In [8]:

traducao_colunas = {
    'id': 'id_imovel',
    'date': 'data',
    'price': 'preco',
    'bedrooms': 'quartos',
    'bathrooms': 'banheiros',
    'sqft_living': 'm2_area_habitavel',    
    'sqft_lot': 'm2_terreno',
    'floors': 'andares',
    'waterfront': 'vista_mar_rio',         
    'view': 'nota_vista',                  
    'condition': 'condicao_imovel',        
    'grade': 'nota_design',                
    'sqft_above': 'm2_acima_solo',          
    'sqft_basement': 'm2_porao',          
    'yr_built': 'ano_construcao',
    'yr_renovated': 'ano_renovacao',
    'zipcode': 'codigo_postal',
    'lat': 'latitude',
    'long': 'longitude',
    'sqft_living15': 'm2_area_vizinhos_15', 
    'sqft_lot15': 'm2_terreno_vizinhos_15' 
}

df_traduzido = df.rename(columns=traducao_colunas)

print(df_traduzido.columns)

Index(['id_imovel', 'data', 'preco', 'quartos', 'banheiros',
       'm2_area_habitavel', 'm2_terreno', 'andares', 'vista_mar_rio',
       'nota_vista', 'condicao_imovel', 'nota_design', 'm2_acima_solo',
       'm2_porao', 'ano_construcao', 'ano_renovacao', 'codigo_postal',
       'latitude', 'longitude', 'm2_area_vizinhos_15',
       'm2_terreno_vizinhos_15'],
      dtype='str')


## 🔹 FASE 2 — Consultas e Agregações

### 4. Análise Histórica de Preços
Preço médio por ano e por mês. O período do dataset (mai/2014 a mai/2015) cobre apenas 13 meses, então a comparação anual reflete meses parciais.

In [9]:
df_traduzido['ano'] = df_traduzido['data'].dt.year
df_traduzido['mes'] = df_traduzido['data'].dt.month

In [10]:
df_media_anual = df_traduzido.groupby('ano')['preco'].mean().reset_index().round(2)

In [11]:
df_media_anual

,ano,preco
0,2014,539182.07
1,2015,541988.99


In [12]:
media_mensal = df_traduzido.groupby(['ano','mes'])['preco'].mean().reset_index().round(2)
media_mensal

,ano,mes,preco
0,2014,5,548080.28
1,2014,6,558002.20
2,2014,7,544788.76
3,2014,8,536445.28
4,2014,9,529253.82
5,2014,10,539031.98
6,2014,11,521961.01
7,2014,12,524461.87
8,2015,1,525870.89
9,2015,2,507851.37


### 5. Análise Localizada
Preço médio, mínimo e máximo de um bairro (CEP) específico dentro de um intervalo de anos definido.

In [13]:
df_traduzido['codigo_postal'].value_counts().reset_index().head()

,codigo_postal,count
0,98103,602
1,98038,590
2,98115,583
3,98052,574
4,98117,553


In [15]:
bairro_escolhido ='98103'
#intevalo de anos:
ano_ini , ano_fim = 2014, 2015


df_bairro = df_traduzido[(df_traduzido['codigo_postal'] == bairro_escolhido) & (df_traduzido['ano'].between(ano_ini, ano_fim)) ]
print(f'O preço médio do cep {bairro_escolhido} é R${df_bairro['preco'].mean():.2f}, o maior preco é {df_bairro['preco'].max():.2f} o menor é {df_bairro['preco'].min()} ')

O preço médio do cep 98103 é R$584919.21, o maior preco é 1695000.00 o menor é 238000.0 


### 6. Ranking de Bairros
Cinco bairros com maior preço médio. Os valores mais altos concentram-se em regiões nobres de King County (ex.: 98039 - Medina).

In [ ]:
media_preco_bairro = (df_traduzido.groupby('codigo_postal')['preco'].mean()
.sort_values(ascending=False)
.reset_index()
.rename(columns= ({'preco' : 'Media_preco','codigo_postal' :'Bairro' }))
.round(2)
)
media_preco_bairro


,Bairro,Media_preco
0,98039,2160606.60
1,98004,1355927.08
2,98040,1194230.02
3,98112,1095499.34
4,98102,901258.27
...,...,...
65,98148,284908.60
66,98001,280804.69
67,98032,251296.24
68,98168,240328.37


In [ ]:
top5 = media_preco_bairro.head(5)
top5

,Bairro,Media_preco
0,98039,2160606.60
1,98004,1355927.08
2,98040,1194230.02
3,98112,1095499.34
4,98102,901258.27


In [ ]:
fig_top5 = px.bar(
    top5,
    x = 'Media_preco',
    y = 'Bairro',
    orientation='h',
    text_auto = '.3s',
    color = 'Media_preco',
    title ='Top 5 bairros com maiores preços médios'
)
fig_top5.update_yaxes(type = 'category')
fig_top5.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 🔹 FASE 3 — Estatística e Outliers

### 7. Identificação de Outliers
Boxplot e histograma da distribuição de preços. Os valores atípicos correspondem a imóveis de luxo reais (mansões, frente-mar), não a inconsistências, e por isso foram mantidos.

In [ ]:
fig_outlier= px.box(df_traduzido, y='preco' , title= 'Distribuição de Preços com Outliers', points ='outliers')
fig_outlier.show()


In [ ]:
fig_hist = px.histogram(df_traduzido , x = 'preco' , nbins= 100 , title ='Distribuição de Preços')
fig_hist.show()

## 🔹 FASE 4 — Visualização e Análise Exploratória

### 8. Relação entre Área e Preço
Dispersão entre área habitável e preço, evidenciando a tendência de crescimento (não perfeitamente linear).

In [ ]:

fig_area = px.scatter(
    df_traduzido, x="m2_area_habitavel", y="preco", title="Relação preço por área"
)
fig_area.show()

### 9. Evolução Temporal dos Preços
Preço médio mês a mês ao longo de todo o período disponível.

In [ ]:

media_mensal['data_formatada'] = media_mensal['ano'].astype(str) + '-' + media_mensal['mes'].astype(str)

fig_preco = px.line(
    media_mensal,
    x='data_formatada',
    y='preco',
    title='Evolução do Preço ao Longo do Tempo',
    labels={'data_formatada': 'Período (Ano-Mês)', 'preco': 'Preço ($)'}
)
fig_preco.update_xaxes(type='category')

fig_preco.show()

## 🔹 FASE 5 — Engenharia de Atributos e Correlações

### 11. Criação de Novas Variáveis
Atributos derivados: densidade de ocupação do lote, idade efetiva (considerando reformas), total de cômodos e indicador de renovação.

In [ ]:
df_traduzido['habitavel_por_terreno'] = df_traduzido['m2_area_habitavel'] / df_traduzido['m2_terreno']
df_traduzido['idade_efetiva'] = df_traduzido['ano'] - df_traduzido[['ano_construcao','ano_renovacao']].max(axis=1)
df_traduzido['total_comodos'] = df_traduzido['quartos'] + df_traduzido['banheiros']
df_traduzido['foi_renovado'] = (df_traduzido['ano_renovacao'] > 0).astype(int)




### 12. Análise de Correlação
Correlação de Pearson (relações lineares e redundância entre features) comparada com Spearman (relações monotônicas), para identificar as variáveis mais influentes e evitar descartar relações não lineares.

In [ ]:
cols_tirar = ['id_imovel', 'data' , 'codigo_postal']
df_correlacao_pearson = df_traduzido.drop(columns= cols_tirar).corr('pearson')


df_correlacao_pearson

In [ ]:
corr_preco_pearson = df_correlacao_pearson['preco'].drop('preco').abs()
corr_preco_spearman = df_traduzido.corr('spearman')['preco'].sort_values(ascending=False).drop('preco').abs()


In [ ]:
comparacao = pd.DataFrame({
    'pearson': corr_preco_pearson,
    'spearman': corr_preco_spearman
})
comparacao['diferenca'] = (comparacao['spearman'] - comparacao['pearson']).round(3)
comparacao['pearson'] = comparacao['pearson'].round(3)
comparacao['spearman'] = comparacao['spearman'].round(3)

comparacao = comparacao.reindex(
    comparacao['spearman'].abs().sort_values(ascending=False).index
)
print(comparacao)

In [ ]:
fig_heatmap = px.imshow(
    df_correlacao_pearson,
    text_auto='.2f',            # escreve o valor em cada célula
    aspect='auto',
    color_continuous_scale='RdBu_r',   # vermelho = +, azul = -
    zmin=-1, zmax=1,            # fixa a escala de cor de -1 a 1
    title='Mapa de calor — correlação de Pearson entre variáveis'
                        )
fig_heatmap.update_layout(height=800, width=900)
fig_heatmap.show()

### 13. Transformação de Variáveis Categóricas
A única categórica relevante para a modelagem seria o código postal (70 níveis, alta cardinalidade). Optou-se por descartá-la, pois a informação de localização é preservada de forma contínua pelas variáveis **latitude** e **longitude** — evitando a explosão dimensional que um one-hot encoding de 70 colunas causaria. As demais variáveis já são numéricas, dispensando codificação.

## 🔹 FASE 6 — Machine Learning

### 14. Modelo de Previsão de Preço

Versão profissional utilizando as bibliotecas do scikit-learn (`Pipeline`, `StandardScaler`, `PolynomialFeatures`, `Ridge`). O pipeline encadeia normalização, geração de termos polinomiais e regressão regularizada, aplicando as transformações apenas com estatísticas do treino para evitar vazamento de dados.

#### Preparação dos dados
Remoção de duplicatas de revenda (evita vazamento entre treino e teste) e separação treino/teste. As features incluem a geografia (latitude e longitude), já que os experimentos anteriores mostraram seu impacto positivo.

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    'm2_area_habitavel', 'nota_design', 'm2_area_vizinhos_15',
    'total_comodos', 'nota_vista', 'm2_porao',
    'habitavel_por_terreno', 'idade_efetiva', 'vista_mar_rio',
    'latitude', 'longitude'
]

# Remove revendas do mesmo imovel (mantem a venda mais recente)
df_modelo = df_traduzido.sort_values('data').drop_duplicates('id_imovel', keep='last')

X = df_modelo[features]
y = df_modelo['preco']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Treino: {X_train.shape[0]} imoveis | Teste: {X_test.shape[0]}')
print(f'Features ({len(features)}):', features)

#### Primeiro pipeline (StandardScaler + PolynomialFeatures + Ridge)
Encadeia normalização, geração de termos polinomiais (com interações) e regressão Ridge num único objeto. O pipeline garante que as transformações sejam ajustadas apenas no treino, evitando vazamento de dados. Este teste inicial usa grau 3 num único split — resultado a ser confirmado por validação cruzada.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error , mean_squared_error , r2_score

modelo = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree = 3 , include_bias = False)),
    ('ridge', Ridge(alpha = 1.0))
])


modelo.fit(X_train , y_train)

y_pred = modelo.predict(X_test)

mae = mean_absolute_error(y_test , y_pred)
rmse = np.sqrt(mean_squared_error(y_test , y_pred))
r2 = r2_score(y_test , y_pred)

print(f"MAE:  US${mae:,.2f}")
print(f"RMSE: US${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

#### Busca de hiperparâmetros com validação cruzada (GridSearchCV)
Testa combinações de grau do polinômio e força de regularização (α) usando validação cruzada de 5 partições. A validação cruzada avalia cada configuração em fatias diferentes do treino, sem tocar no conjunto de teste — a forma metodologicamente correta de escolher hiperparâmetros.

In [ ]:
from sklearn.model_selection import GridSearchCV

modelo = Pipeline([
    ('scaler' , StandardScaler()),
    ('poly' , PolynomialFeatures(include_bias = False)),
    ('ridge' , Ridge())
])


param_grid = {
    'poly__degree': [2,3],
    'ridge__alpha' : [0.1 , 1.0 ,10.0, 100.0]
}

busca = GridSearchCV(
    modelo,
    param_grid,
    cv =5,
    scoring = 'r2',
    n_jobs = -1
)

busca.fit(X_train,y_train)


print("Melhor combinação:", busca.best_params_)
print(f"Melhor R² (validação cruzada): {busca.best_score_:.4f}")

In [ ]:
# O GridSearch já treina o melhor modelo no treino inteiro — é só usar
y_pred = busca.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Desempenho no TESTE (dados nunca vistos):")
print(f"MAE:  US${mae:,.2f}")
print(f"RMSE: US${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

#### Refinamento da regularização
Como a primeira busca escolheu o maior α disponível, testam-se valores maiores para confirmar o ponto ótimo. O desempenho estabiliza em torno de R² 0.79, indicando que o modelo atingiu um platô — regularização adicional não altera o resultado.

In [ ]:
param_grid = {
    'poly__degree': [2, 3],
    'ridge__alpha': [100.0, 300.0, 1000.0, 3000.0]
}

busca = GridSearchCV(modelo, param_grid, cv=5, scoring='r2', n_jobs=-1)
busca.fit(X_train, y_train)

print("Melhor combinação:", busca.best_params_)
print(f"Melhor R² (CV): {busca.best_score_:.4f}")

y_pred = busca.predict(X_test)
print(f"R² teste: {r2_score(y_test, y_pred):.4f}")
print(f"MAE teste: US${mean_absolute_error(y_test, y_pred):,.2f}")

#### Variáveis mais influentes no preço
Como as features foram padronizadas, os coeficientes do Ridge são comparáveis entre si: quanto maior o valor absoluto, maior a influência da variável no preço. Área habitável, nota de design e latitude aparecem como os fatores dominantes — confirmando que tamanho, qualidade e localização são os principais determinantes.

In [ ]:
# Extrai o modelo final treinado de dentro do pipeline vencedor
melhor_pipeline = busca.best_estimator_
ridge_final = melhor_pipeline.named_steps['ridge']
poly_final = melhor_pipeline.named_steps['poly']

# Nomes dos termos polinomiais (inclui interações)
nomes = poly_final.get_feature_names_out(features)

# Monta tabela de coeficientes ordenada por influência (valor absoluto)
coefs = pd.DataFrame({
    'termo': nomes,
    'coeficiente': ridge_final.coef_
})
coefs['abs'] = coefs['coeficiente'].abs()
coefs = coefs.sort_values('abs', ascending=False).drop(columns='abs')

print("Top 15 termos mais influentes no preço:")
print(coefs.head(15).to_string(index=False))

### 16. Agrupamento de Imóveis ou Bairros

Agrupamento não supervisionado dos imóveis por comportamento de preço e características, usando K-Means. Diferente da modelagem anterior (supervisionada), aqui não há alvo a prever — o algoritmo descobre grupos naturais de imóveis semelhantes. O número de grupos é definido pelo método do cotovelo.

#### Método do cotovelo — escolha do número de clusters
Testa K de 1 a 10 e mede a inércia (dispersão interna dos grupos). O ponto onde a curva deixa de cair acentuadamente — o "cotovelo" — indica o K ideal. Aqui, K=4.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Features que descrevem o imovel (o preco PODE entrar: nao ha previsao, so agrupamento)
features_cluster = [
    'preco', 'm2_area_habitavel', 'total_comodos',
    'nota_design', 'latitude', 'longitude', 'idade_efetiva'
]

X_cluster = df_modelo[features_cluster]

# Padroniza — essencial, pois o K-Means usa distancia
scaler_c = StandardScaler()
X_cluster_norm = scaler_c.fit_transform(X_cluster)

# Metodo do cotovelo
inercias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_norm)
    inercias.append(km.inertia_)

fig_cotovelo = px.line(x=list(range(1, 11)), y=inercias, markers=True,
                       title='Método do Cotovelo')
fig_cotovelo.update_xaxes(title='Número de clusters (K)')
fig_cotovelo.update_yaxes(title='Inércia')
fig_cotovelo.show()

#### K-Means com 4 grupos e perfil de cada segmento
Aplica o K-Means definitivo e calcula o "imóvel típico" de cada grupo (médias nas features originais, para facilitar a interpretação).

In [ ]:
# K-Means definitivo com 4 grupos
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_modelo['cluster'] = kmeans.fit_predict(X_cluster_norm)

# Perfil medio de cada grupo (features originais, nao padronizadas)
perfil = df_modelo.groupby('cluster')[features_cluster].mean().round(2)
perfil['qtd_imoveis'] = df_modelo['cluster'].value_counts().sort_index()
print(perfil.to_string())

**Interpretação dos grupos.** Os quatro segmentos são coerentes e refletem a lógica do mercado imobiliário:

- **Cluster 0 — Populares antigos** (6.049): menor área (130 m²), design mais baixo e, sobretudo, os mais velhos (73 anos). As casas mais modestas.
- **Cluster 1 — Alto luxo** (3.123): preço mais que o dobro dos demais (US$1,11 mi), área ampla (339 m²), melhor design e imóveis novos.
- **Cluster 2 — Alto padrão** (6.575): bons imóveis ao norte (lat 47.65), preço médio de US$561 mil.
- **Cluster 3 — Medianos ao sul** (5.689): área semelhante à do cluster 2, mas ao sul (lat 47.39) e mais baratos (US$332 mil).

A validade dos grupos se confirma pela separação entre os clusters 2 e 3: imóveis de área parecida, porém em latitudes distintas e com preços muito diferentes — o algoritmo capturou a influência da **localização** sem supervisão, coerente com os fatores identificados na análise de correlação e nos coeficientes do modelo.

## 🔹 FASE 7 — Deep Learning

### 17. Preparação dos Dados para Rede Neural


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Features da rede neural (mesmas do modelo supervisionado, com geografia)
features_nn = [
    'm2_area_habitavel', 'nota_design', 'm2_area_vizinhos_15',
    'total_comodos', 'nota_vista', 'm2_porao',
    'habitavel_por_terreno', 'idade_efetiva', 'vista_mar_rio',
    'latitude', 'longitude'
]

nn_X = df_modelo[features_nn].values
nn_y = df_modelo['preco'].values

# 1º split: separa o TESTE (20%) do resto (80%)
nn_X_temp, nn_X_test, nn_y_temp, nn_y_test = train_test_split(
    nn_X, nn_y, test_size=0.2, random_state=42
)

# 2º split: divide o resto em TREINO (60%) e VALIDAÇÃO (20%)
nn_X_train, nn_X_val, nn_y_train, nn_y_val = train_test_split(
    nn_X_temp, nn_y_temp, test_size=0.25, random_state=42
)

print(f"Treino:    {nn_X_train.shape[0]}")
print(f"Validação: {nn_X_val.shape[0]}")
print(f"Teste:     {nn_X_test.shape[0]}")

# --- Padronização das FEATURES (estatísticas só do treino) ---
nn_scaler_X = StandardScaler()
nn_X_train_s = nn_scaler_X.fit_transform(nn_X_train)
nn_X_val_s   = nn_scaler_X.transform(nn_X_val)
nn_X_test_s  = nn_scaler_X.transform(nn_X_test)

# --- Padronização do ALVO (z-score à mão, para facilitar a reversão) ---
nn_mu_y, nn_sigma_y = nn_y_train.mean(), nn_y_train.std()
nn_y_train_s = (nn_y_train - nn_mu_y) / nn_sigma_y
nn_y_val_s   = (nn_y_val - nn_mu_y) / nn_sigma_y
nn_y_test_s  = (nn_y_test - nn_mu_y) / nn_sigma_y

print("\nDados preparados e padronizados.")

### 18. Construção da Rede Neural


### 19. Treinamento e Avaliação do Modelo